# nano-gpt-lab: universal Colab / Kaggle runner

Same notebook, both platforms. Two-tier workflow:
- **ZBook (local)**: first-principles build, unit tests, tiny models, debugging (run this notebook locally too).
- **Colab/Kaggle T4**: larger NanoGPT, TinyStories, OpenWebText subset, FlashAttention benchmarks, GPT-2 ~124M.

**Colab**: outputs mirror to `/content/drive/MyDrive/nano-gpt-lab/`
**Kaggle**: outputs persist automatically in `/kaggle/working/nano-gpt-lab/` (saved to your Kaggle account as the notebook output)

In [ ]:
# --- 0. WHERE ARE WE? ------------------------------------------------------
import os, sys, zipfile, shutil, glob
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""
print("colab:", IN_COLAB, "| kaggle:", IN_KAGGLE, "| local:", not (IN_COLAB or IN_KAGGLE))

In [ ]:
# --- 1. GET THE REPO ------------------------------------------------------
# Locally: the repo is the parent of this notebook's folder - nothing to do.
# Colab   : run this cell, upload nano-gpt-lab.zip (the whole repo zipped).
# Kaggle  : add nano-gpt-lab.zip as a DATASET input first, then run this.
REPO_NAME = "nano-gpt-lab"

def find_repo():
    here = os.path.abspath("")
    for cand in [here, os.path.dirname(here),
                 os.path.join(here, REPO_NAME),
                 os.path.join(os.path.dirname(here), REPO_NAME)]:
        if os.path.exists(os.path.join(cand, "scripts", "train.py")):
            return cand
    if IN_KAGGLE:
        zips = glob.glob("/kaggle/input/**/nano-gpt-lab.zip", recursive=True)              + glob.glob("/kaggle/input/**/*nano*.zip", recursive=True)
        if zips:
            dest = f"/kaggle/working/{REPO_NAME}"
            with zipfile.ZipFile(zips[0]) as z:
                z.extractall(dest)
            inner = os.path.join(dest, REPO_NAME)
            return inner if os.path.exists(os.path.join(inner, "scripts")) else dest
    return None

repo = find_repo()
if repo is None and IN_COLAB:
    from google.colab import files
    print("upload nano-gpt-lab.zip:")
    files.upload()
    dest = f"/content/{REPO_NAME}"
    for zf in glob.glob("/content/nano-gpt-lab.zip"):
        with zipfile.ZipFile(zf) as z:
            z.extractall(dest)
        inner = os.path.join(dest, REPO_NAME)
        if os.path.exists(os.path.join(inner, "scripts")):
            dest = inner
    repo = dest if os.path.exists(os.path.join(dest, "scripts", "train.py")) else None
assert repo, "could not find the repo (Colab: upload the zip; Kaggle: add it as a Dataset)"
print("repo at:", repo)
os.chdir(repo)
sys.path.insert(0, repo)

In [ ]:
# --- 2. DEPS + ENVIRONMENT ------------------------------------------------
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyyaml", "tqdm", "matplotlib", "requests"],
               capture_output=True)
from src.utils.environment import print_environment
print_environment()

In [ ]:
# --- 3. SMOKE TESTS (always cheap, always worth it) ------------------------
%run scripts/run_tests.py

In [ ]:
# --- 4. SETTINGS - edit these three lines ---------------------------------
DATASET = "shakespeare"        # shakespeare | tinystories | openwebtext
SIZE    = "gpt2"               # nano (ZBook) | small | gpt2 | gpt3
STEPS   = 2000                 # training steps per variant
SEEDS   = "42"                 # "42 123 2026" for the full seed set

# --- RUN THE WHOLE A/B/C/D COMPARISON (identical recipe, one variable) -----
%run scripts/train.py --dataset {DATASET} --compare --size {SIZE} \
    --steps {STEPS} --seeds {SEEDS}

In [ ]:
# --- 5. THE LAYMAN REPORTS -------------------------------------------------
%run scripts/analyze.py --dataset {DATASET} --size {SIZE}

from IPython.display import Image, display
import glob
from src.utils.environment import repo_root
for png in sorted(glob.glob(os.path.join(repo_root(), "results", DATASET, "*.png"))):
    print(os.path.basename(png))
    display(Image(png))

**Where your work went:**

- Colab: `/content/drive/MyDrive/nano-gpt-lab/runs/...` (checkpoints mirrored to Drive every checkpoint interval + at the end).
- Kaggle: `/kaggle/working/nano-gpt-lab/runs/...` - persisted to your Kaggle account automatically as the notebook output.
- Local: `runs/` inside the repo.

Resume after a disconnect: same command with `--resume`.
TinyStories next: set DATASET above to `tinystories` (downloads a 50MB documented subset).